# Notebook 11: DSPy — Programming, Not Prompting

**DSPy** is a framework for *programming* with language models, rather than *prompting* them.
Created by **Omar Khattab** — who is also a co-author of the RLM paper — DSPy shares a
deep philosophical connection with Recursive Language Models.

Both DSPy and RLM treat LLMs as **programmable components** rather than black-box chat interfaces.
The key difference lies in *who* does the programming:

- **DSPy**: The developer composes predefined modules; an optimizer tunes the prompts automatically.
- **RLM**: The model itself writes and composes programs at inference time.

In this notebook you will learn:
- What DSPy is and how its core abstractions work (Signatures, Modules, Optimizers)
- How to build DSPy-style programs for question answering
- How DSPy and RLM compare on the same multi-hop QA task
- Where each approach sits on the LLM autonomy spectrum

## What is DSPy?

DSPy replaces hand-written prompts with three core abstractions:

| Abstraction | Purpose | Analogy |
|-------------|---------|--------|
| **Signatures** | Define input/output specs for an LLM call | Function type signatures |
| **Modules** | Composable LLM programs (ChainOfThought, ReAct, etc.) | Neural network layers |
| **Optimizers** | Auto-tune prompts and few-shot examples to maximize a metric | Backpropagation / hyperparameter search |

### The Key Insight

Instead of writing:
```
"You are a helpful assistant. Think step by step. Given the context, answer..."
```

You write:
```python
class QA(dspy.Signature):
    context = dspy.InputField()
    question = dspy.InputField()
    answer = dspy.OutputField()

qa = dspy.ChainOfThought(QA)
result = qa(context=my_doc, question="What is the secret code?")
```

DSPy handles the prompt engineering for you. The optimizer searches for prompts and
few-shot examples that maximize your evaluation metric on a training set.

## Setup

Since DSPy requires a live LLM backend, we build a `SimulatedDSPyLM` that mimics DSPy's
interface using pattern matching — the same approach used throughout these notebooks.
We also show what the real DSPy code would look like alongside each simulation.

In [ ]:
import sys
sys.path.insert(0, "..")
import json, os, re, textwrap

from rlm_core import RLMEngine, Sandbox
from rlm_core.visualizer import tree_to_text

# ---------------------------------------------------------------------------
# Simulated DSPy LM — mirrors real DSPy's dspy.LM interface
# ---------------------------------------------------------------------------
class SimulatedDSPyLM:
    """Simulates a DSPy language model for demonstrations.
    
    In real DSPy you would write:
        lm = dspy.LM('openai/gpt-4o-mini')  # or a local vLLM model
        dspy.configure(lm=lm)
    
    This class provides the same external behaviour using pattern matching
    so the notebook runs without a live LLM server.
    """
    
    def __init__(self):
        self.call_count = 0
        self.history = []  # Track all calls for inspection
    
    def __call__(self, prompt, **kwargs):
        """Simulate an LLM call — returns a list of strings (DSPy convention)."""
        self.call_count += 1
        self.history.append(prompt if isinstance(prompt, str) else str(prompt))
        text = self._route(prompt if isinstance(prompt, str) else str(prompt))
        return [text]
    
    def _route(self, prompt):
        lower = prompt.lower()
        
        # --- Chain-of-Thought QA about secret code ---
        if "secret" in lower and "code" in lower:
            return (
                "Reasoning: Let me search through the context for any mention of a "
                "secret code. I find the sentence 'The secret code is BLUE-FALCON-42' "
                "embedded in the text.\n"
                "Answer: BLUE-FALCON-42"
            )
        
        # --- Multi-hop: SolarFlow battery CTO ---
        if "solarflow" in lower and "cto" in lower:
            return (
                "Reasoning: First I need to find who makes the SolarFlow battery. "
                "From the context, NovaTech designed the SolarFlow battery. "
                "Then I need to find NovaTech's CTO. "
                "The context states Sarah Martinez is the CTO of NovaTech.\n"
                "Answer: Sarah Martinez"
            )
        
        # --- Multi-hop: NovaPad lead engineer ---
        if ("novapad" in lower or "nova pad" in lower) and ("engineer" in lower or "who" in lower):
            return (
                "Reasoning: The NovaPad was developed by TechVista Inc. "
                "The lead engineer at TechVista is Dr. James Park.\n"
                "Answer: Dr. James Park"
            )
        
        # --- Hop 1: Who made the NovaPad? ---
        if "novapad" in lower and "who" not in lower and "company" in lower:
            return "TechVista Inc"
        
        # --- Hop 2: Lead engineer at TechVista? ---
        if "techvista" in lower and "engineer" in lower:
            return "Dr. James Park"
        
        # --- Summarization / general ---
        if "summarize" in lower or "summary" in lower:
            return (
                "Reasoning: The context discusses technology companies and products.\n"
                "Answer: The documents describe TechVista Inc, its NovaPad tablet, "
                "and key personnel including Dr. James Park."
            )
        
        # --- Aggregation ---
        if "how many" in lower or "count" in lower:
            return (
                "Reasoning: I need to count items matching the criteria in the data.\n"
                "Answer: Based on the available context, I found the matching items."
            )
        
        # Default
        return (
            "Reasoning: Let me analyze the context to find relevant information.\n"
            "Answer: Information extracted from context."
        )


# ---------------------------------------------------------------------------
# Simulated RLM client (same pattern as notebooks 9-10)
# ---------------------------------------------------------------------------
class SimulatedClient:
    def __init__(self):
        self.call_count = 0
        self.total_prompt_tokens = 0
        self.total_completion_tokens = 0
    
    def completion(self, prompt, **kwargs):
        self.call_count += 1
        self.total_prompt_tokens += len(prompt.split())
        self.total_completion_tokens += 15
        
        class Result:
            def __init__(self, text, pt, ct):
                self.text, self.prompt_tokens, self.completion_tokens = text, pt, ct
        
        lower = prompt.lower()
        
        if "secret" in lower or "code" in lower:
            text = '''import re\nmatch = re.search(r'secret code is ([A-Z0-9-]+)', context)\nFINAL(match.group(1) if match else "Not found")'''
        elif ("novapad" in lower or "nova pad" in lower) and ("engineer" in lower or "who" in lower):
            text = '''# Multi-hop: find who made NovaPad, then find their lead engineer\nimport re\n# Step 1: Find company\ncompany_match = re.search(r'(\\w+ \\w+) .*(?:developed|made|created).*NovaPad|NovaPad.*(?:developed|made|created) by (\\w+ \\w+)', context)\ncompany = "TechVista Inc"  # extracted from context\n# Step 2: Find lead engineer at that company\nengineer_match = re.search(r'(Dr\\.? [A-Z][a-z]+ [A-Z][a-z]+) is the lead engineer', context)\nFINAL(engineer_match.group(1) if engineer_match else "Not found")'''
        elif "how many" in lower or "count" in lower or "red" in lower:
            text = '''import json\ndata = json.loads(context) if context.strip().startswith('[') else []\ncount = sum(1 for i in data if i.get("color")=="red" and i.get("size")=="large")\nFINAL(f"{count}")'''
        elif "engineer" in lower or "who" in lower:
            text = '''import re\nm = re.search(r'(Dr\\.? [A-Z][a-z]+ [A-Z][a-z]+).*(?:lead engineer|CTO)', context)\nFINAL(m.group(1) if m else "Not found")'''
        else:
            text = 'FINAL("Information processed")'
        
        return Result(text, len(prompt.split()), 15)


# Instantiate both
dspy_lm = SimulatedDSPyLM()
client = SimulatedClient()
engine = RLMEngine(client=client, max_depth=3)

# Load datasets
with open("../data/samples/needle_haystack.txt") as f:
    needle_doc = f.read()
docs = []
doc_dir = "../data/samples/multihop_docs"
for fname in sorted(os.listdir(doc_dir)):
    if fname.endswith('.txt'):
        with open(os.path.join(doc_dir, fname)) as f:
            docs.append(f.read())

print(f"SimulatedDSPyLM ready  (mimics dspy.LM interface)")
print(f"SimulatedClient ready  (for RLM engine)")
print(f"Loaded: needle ({len(needle_doc)} chars), {len(docs)} multi-hop docs")

## DSPy Signatures — Declaring LLM Input/Output

A **Signature** in DSPy is like a function type signature. It declares what goes
into an LLM call and what comes out — without specifying *how* the LLM should
produce the output. DSPy handles the prompt generation.

```python
# Real DSPy code (requires dspy installed + live LLM):
class QA(dspy.Signature):
    """Answer a question based on the given context."""
    context = dspy.InputField(desc="Relevant documents or passages")
    question = dspy.InputField(desc="The question to answer")
    answer = dspy.OutputField(desc="A concise factual answer")
```

Below we simulate this pattern so you can see the structure without installing DSPy.

In [ ]:
# ---------------------------------------------------------------------------
# Simulated DSPy Signatures and Modules
# ---------------------------------------------------------------------------
class SimulatedSignature:
    """Mimics dspy.Signature — declares input/output fields."""
    def __init__(self, name, input_fields, output_fields, doc=""):
        self.name = name
        self.input_fields = input_fields   # list of (name, desc)
        self.output_fields = output_fields  # list of (name, desc)
        self.doc = doc
    
    def __repr__(self):
        ins = ", ".join(f"{n}" for n, _ in self.input_fields)
        outs = ", ".join(f"{n}" for n, _ in self.output_fields)
        return f"{self.name}({ins} -> {outs})"


# Define our signatures
QA_sig = SimulatedSignature(
    name="QA",
    input_fields=[("context", "Relevant documents"), ("question", "The question")],
    output_fields=[("answer", "A concise factual answer")],
    doc="Answer a question based on the given context."
)

QA_with_reasoning_sig = SimulatedSignature(
    name="QAWithReasoning",
    input_fields=[("context", "Relevant documents"), ("question", "The question")],
    output_fields=[("reasoning", "Step-by-step reasoning"), ("answer", "Final answer")],
    doc="Answer a question with explicit chain-of-thought reasoning."
)

MultiHop_sig = SimulatedSignature(
    name="MultiHopQA",
    input_fields=[("context", "Multiple documents"), ("question", "A multi-hop question")],
    output_fields=[
        ("hop1_query", "First retrieval query"),
        ("hop1_result", "First intermediate finding"),
        ("hop2_query", "Second retrieval query"),
        ("hop2_result", "Second intermediate finding"),
        ("answer", "Final composed answer"),
    ],
    doc="Answer a question that requires connecting facts across multiple documents."
)

print("DSPy Signatures defined:")
print(f"  {QA_sig}")
print(f"  {QA_with_reasoning_sig}")
print(f"  {MultiHop_sig}")
print()
print("Signatures declare WHAT the LLM should do, not HOW.")
print("DSPy generates the prompt automatically from the signature.")

## DSPy Modules — Composable LLM Programs

A **Module** in DSPy wraps a signature with a specific prompting strategy.
The most commonly used modules are:

| Module | Strategy |
|--------|----------|
| `dspy.Predict` | Direct input-to-output (zero-shot) |
| `dspy.ChainOfThought` | Adds "Reasoning: ..." before the answer |
| `dspy.ReAct` | Thought-Action-Observation loop |
| `dspy.ProgramOfThought` | Generates code to compute the answer |

Modules are **composable** — you can nest them inside each other, just like
PyTorch layers inside an `nn.Module`.

```python
# Real DSPy code:
cot_qa = dspy.ChainOfThought(QA)            # wraps QA signature with CoT
result = cot_qa(context=doc, question=q)     # returns Prediction(reasoning=..., answer=...)
```

In [ ]:
# ---------------------------------------------------------------------------
# Simulated DSPy Modules
# ---------------------------------------------------------------------------
class SimulatedPrediction:
    """Mimics dspy.Prediction — a structured output from a module."""
    def __init__(self, **kwargs):
        for k, v in kwargs.items():
            setattr(self, k, v)
        self._fields = kwargs
    
    def __repr__(self):
        items = ", ".join(f"{k}={v!r:.60}" for k, v in self._fields.items())
        return f"Prediction({items})"


class SimulatedChainOfThought:
    """Mimics dspy.ChainOfThought — adds reasoning before the answer.
    
    Real DSPy equivalent:
        cot = dspy.ChainOfThought('context, question -> answer')
        result = cot(context=..., question=...)
        print(result.reasoning, result.answer)
    """
    
    def __init__(self, signature, lm):
        self.signature = signature
        self.lm = lm
    
    def __call__(self, **kwargs):
        # Build prompt from signature + inputs
        prompt_parts = [f"{self.signature.doc}\n"]
        for name, desc in self.signature.input_fields:
            prompt_parts.append(f"{name}: {kwargs.get(name, '')}")
        prompt_parts.append("Produce: reasoning, then answer.")
        prompt = "\n".join(prompt_parts)
        
        # Call LM
        response = self.lm(prompt)[0]
        
        # Parse structured output
        reasoning = ""
        answer = response
        if "Reasoning:" in response:
            parts = response.split("Answer:")
            reasoning = parts[0].replace("Reasoning:", "").strip()
            answer = parts[1].strip() if len(parts) > 1 else ""
        
        return SimulatedPrediction(reasoning=reasoning, answer=answer)


# Build a ChainOfThought module
cot_qa = SimulatedChainOfThought(QA_with_reasoning_sig, dspy_lm)

# Run it on the needle-in-a-haystack task
result = cot_qa(context=needle_doc[:500], question="What is the secret code?")

print("=== DSPy ChainOfThought Result ===")
print(f"Reasoning: {result.reasoning}")
print(f"Answer:    {result.answer}")
print(f"\nLM calls:  {dspy_lm.call_count}")

## DSPy Optimizers — Auto-Tuning Prompts

The most powerful idea in DSPy is the **optimizer**. Instead of manually
engineering prompts, you define:

1. A **metric** (e.g., exact match, F1 score)
2. A **training set** of (input, expected_output) examples
3. An **optimizer** (BootstrapFewShot, MIPRO, etc.)

The optimizer then searches over prompt formulations and few-shot examples to
maximize your metric — analogous to how backpropagation optimizes neural network weights.

```python
# Real DSPy code:
from dspy.teleprompt import BootstrapFewShot

def exact_match(example, pred):
    return example.answer.lower() == pred.answer.lower()

optimizer = BootstrapFewShot(metric=exact_match, max_bootstrapped_demos=4)
optimized_cot = optimizer.compile(cot_qa, trainset=train_examples)
```

Below we simulate this process to show the concept.

In [ ]:
# ---------------------------------------------------------------------------
# Simulated DSPy Optimizer
# ---------------------------------------------------------------------------
class SimulatedBootstrapFewShot:
    """Mimics dspy.teleprompt.BootstrapFewShot.
    
    Real DSPy searches over prompt formulations and few-shot demos.
    We simulate the before/after to show the concept.
    """
    
    def __init__(self, metric, max_bootstrapped_demos=4):
        self.metric = metric
        self.max_demos = max_bootstrapped_demos
        self.log = []
    
    def compile(self, module, trainset):
        """Simulate the optimization process."""
        print(f"Optimizing with {len(trainset)} training examples...")
        print(f"Max bootstrapped demos: {self.max_demos}")
        print()
        
        # Evaluate before optimization
        correct_before = 0
        for example in trainset:
            pred = module(**{k: v for k, v in example.items() if k != 'expected'})
            match = self.metric(example, pred)
            self.log.append({"example": example["question"][:50], "match": match})
            correct_before += int(match)
        
        before_acc = correct_before / len(trainset)
        print(f"Before optimization: {before_acc:.0%} accuracy")
        
        # Simulate optimization (in real DSPy this searches over prompts)
        after_acc = min(1.0, before_acc + 0.25)  # Simulated improvement
        print(f"After optimization:  {after_acc:.0%} accuracy")
        print(f"Selected {min(self.max_demos, len(trainset))} few-shot demos")
        print()
        print("In real DSPy, the optimized module now includes:")
        print("  - Refined instruction phrasing")
        print("  - Curated few-shot examples in the prompt")
        print("  - Potentially different prompt templates")
        
        return module  # Return (in simulation, same module)


# Define training examples
train_examples = [
    {"context": needle_doc[:500], "question": "What is the secret code?",
     "expected": "BLUE-FALCON-42"},
    {"context": docs[1], "question": "Who developed the NovaPad?",
     "expected": "TechVista Inc"},
    {"context": docs[2], "question": "Who is the lead engineer at TechVista?",
     "expected": "Dr. James Park"},
]

# Define metric
def exact_match(example, pred):
    return example["expected"].lower() in pred.answer.lower()

# Run optimizer
optimizer = SimulatedBootstrapFewShot(metric=exact_match, max_bootstrapped_demos=4)
optimized_qa = optimizer.compile(cot_qa, trainset=train_examples)

## DSPy vs RLM Philosophy

Omar Khattab co-authored both DSPy and the RLM paper. This is not a coincidence —
both frameworks share a core belief:

> **LLMs should be treated as programmable functions, not chatbots.**

But they take different paths from this shared starting point:

| Dimension | DSPy | RLM |
|-----------|------|-----|
| **Who writes the program?** | The developer | The model itself |
| **When is the program composed?** | Design time (before deployment) | Inference time (per query) |
| **Module structure** | Predefined (ChainOfThought, ReAct, ...) | Emergent (model decides what code to write) |
| **Optimization** | Offline optimizer tunes prompts | Model adapts strategy per input |
| **Composability** | Developer nests modules explicitly | Model creates its own call graph |
| **Reproducibility** | High (fixed pipeline) | Variable (depends on model's code) |

### The Complementary Vision

Think of DSPy and RLM as two ends of a control spectrum:

- **DSPy** gives the developer maximum control: "I design the pipeline structure,
  and DSPy optimizes the prompts within that structure."
- **RLM** gives the model maximum autonomy: "I give the model a sandbox and a task,
  and it decides how to decompose and solve it."

Neither is universally better. The right choice depends on whether you need
**predictability** (DSPy) or **adaptability** (RLM).

## Same Task, Both Approaches — Multi-Hop QA

Let us solve the same multi-hop question with both DSPy and RLM:

> *"Who is the lead engineer at the company that developed the NovaPad?"*

This requires two hops:
1. NovaPad was developed by **TechVista Inc** (from doc_02)
2. The lead engineer at TechVista is **Dr. James Park** (from doc_03)

In [ ]:
# ---------------------------------------------------------------------------
# DSPy Approach: Composed Modules with Explicit Pipeline
# ---------------------------------------------------------------------------
class SimulatedMultiHopDSPy:
    """Mimics a DSPy multi-hop QA pipeline.
    
    Real DSPy equivalent:
        class MultiHopQA(dspy.Module):
            def __init__(self):
                self.hop1 = dspy.ChainOfThought('context, question -> intermediate_answer')
                self.hop2 = dspy.ChainOfThought('context, question, intermediate_answer -> answer')
            
            def forward(self, context, question):
                hop1_result = self.hop1(context=context, question=question)
                hop2_result = self.hop2(
                    context=context, question=question,
                    intermediate_answer=hop1_result.intermediate_answer
                )
                return hop2_result
    """
    
    def __init__(self, lm):
        self.lm = lm
        self.trace = []
    
    def forward(self, context, question):
        self.trace = []
        
        # Hop 1: Decompose the question
        hop1_prompt = f"Context: {context[:1000]}\nQuestion: {question}\nWhat company is mentioned?"
        hop1_response = self.lm(hop1_prompt)[0]
        self.trace.append({"hop": 1, "query": "What company developed the NovaPad?",
                           "result": "TechVista Inc"})
        
        # Hop 2: Use intermediate result to answer the full question
        hop2_prompt = (f"Context: {context[:1000]}\n"
                       f"We know: NovaPad was made by TechVista Inc.\n"
                       f"Question: Who is the lead engineer at TechVista Inc?")
        hop2_response = self.lm(hop2_prompt)[0]
        self.trace.append({"hop": 2, "query": "Who is the lead engineer at TechVista?",
                           "result": "Dr. James Park"})
        
        return SimulatedPrediction(
            hop1="TechVista Inc",
            hop2="Dr. James Park",
            answer="Dr. James Park"
        )


# Run DSPy approach
multi_context = "\n\n---\n\n".join(docs)
question = "Who is the lead engineer at the company that developed the NovaPad?"

dspy_multi = SimulatedMultiHopDSPy(dspy_lm)
dspy_result = dspy_multi.forward(context=multi_context, question=question)

print("=" * 60)
print("DSPy Multi-Hop Pipeline")
print("=" * 60)
for step in dspy_multi.trace:
    print(f"  Hop {step['hop']}: {step['query']}")
    print(f"         -> {step['result']}")
print(f"\n  Final answer: {dspy_result.answer}")
print(f"  LM calls: {2} (one per hop — predetermined pipeline)")

In [ ]:
# ---------------------------------------------------------------------------
# RLM Approach: Model Writes Its Own Program
# ---------------------------------------------------------------------------
print("=" * 60)
print("RLM Self-Programmed Approach")
print("=" * 60)

rlm_result = engine.run(question, multi_context)

print(f"\n  Code written by the model:")
print(textwrap.indent(rlm_result.root_node.code_executed, "    "))
print(f"\n  Final answer: {rlm_result.answer}")
print(f"  LLM calls: {rlm_result.total_llm_calls}")
print(f"  Max depth: {rlm_result.max_depth_reached}")
print()
print("Recursion tree:")
print(tree_to_text(rlm_result.root_node))

In [ ]:
# ---------------------------------------------------------------------------
# Side-by-Side Comparison
# ---------------------------------------------------------------------------
print("=" * 60)
print("Side-by-Side Comparison")
print("=" * 60)
print()
print(f"Question: {question}")
print(f"Expected: Dr. James Park")
print()
print(f"{'':>20} {'DSPy':>15} {'RLM':>15}")
print(f"{'':>20} {'-'*15} {'-'*15}")
print(f"{'Answer':>20} {dspy_result.answer:>15} {rlm_result.answer:>15}")
print(f"{'LLM calls':>20} {'2':>15} {str(rlm_result.total_llm_calls):>15}")
print(f"{'Pipeline design':>20} {'Developer':>15} {'Model':>15}")
print(f"{'Hop structure':>20} {'Predetermined':>15} {'Emergent':>15}")
print(f"{'Adaptable to':>20} {'similar tasks':>15} {'any task':>15}")
print()
print("Both got the right answer, but through fundamentally different paths:")
print("  DSPy: Developer designed a 2-hop pipeline; optimizer tuned the prompts.")
print("  RLM:  Model wrote a regex-based program to extract the answer directly.")

## The Spectrum: Prompting, DSPy, and RLM

There is a natural spectrum of how much autonomy we give the language model:

```
Less autonomy                                              More autonomy
|------------|---------------------|------------------------|-----------|
Prompting    Few-shot / CoT        DSPy (structured        RLM (self-
(manual)     (manual examples)     modules + optimizer)     programming)
```

Each step to the right hands more responsibility to the model:

1. **Prompting**: The developer writes the complete prompt by hand.
2. **Few-shot / Chain-of-Thought**: The developer provides examples; the model follows the pattern.
3. **DSPy**: The developer defines the module structure; the optimizer writes the prompts.
4. **RLM**: The model writes its own program — structure, logic, and sub-calls.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

fig, ax = plt.subplots(figsize=(14, 6))

# Define the spectrum
approaches = [
    ("Manual\nPrompting", 0.12, "#FF6B6B",
     "Developer writes\nthe entire prompt"),
    ("Few-shot /\nCoT", 0.34, "#FFA07A",
     "Developer provides\nexamples & pattern"),
    ("DSPy\n(Modules +\nOptimizer)", 0.58, "#4ECDC4",
     "Developer designs\nstructure; optimizer\ntunes prompts"),
    ("RLM\n(Self-\nProgramming)", 0.84, "#45B7D1",
     "Model writes its\nown programs at\ninference time"),
]

# Draw the gradient bar
gradient = np.linspace(0, 1, 256).reshape(1, -1)
ax.imshow(gradient, aspect='auto', cmap='RdYlGn', extent=[0, 1, 0.35, 0.65],
          alpha=0.3)
ax.add_patch(plt.Rectangle((0, 0.35), 1, 0.3, fill=False, edgecolor='gray',
                            linewidth=1.5))

# Place each approach
for label, x, color, desc in approaches:
    # Circle on the bar
    ax.plot(x, 0.5, 'o', color=color, markersize=20, markeredgecolor='black',
            markeredgewidth=1.5, zorder=5)
    # Label above
    ax.text(x, 0.78, label, ha='center', va='bottom', fontsize=11,
            fontweight='bold', color=color)
    # Description below
    ax.text(x, 0.22, desc, ha='center', va='top', fontsize=8.5,
            color='#444444', style='italic')

# Arrows and labels for the spectrum axis
ax.annotate('', xy=(0.98, 0.5), xytext=(0.02, 0.5),
            arrowprops=dict(arrowstyle='->', color='black', lw=2))
ax.text(0.0, 0.5, 'Developer\nControl', ha='right', va='center',
        fontsize=10, fontweight='bold', color='#CC3333')
ax.text(1.0, 0.5, 'Model\nAutonomy', ha='left', va='center',
        fontsize=10, fontweight='bold', color='#2277AA')

# Title
ax.set_title('The LLM Autonomy Spectrum: From Prompting to Self-Programming',
             fontsize=14, fontweight='bold', pad=20)

# Highlight the DSPy-RLM connection
ax.annotate('Shared philosophy:\n"LLMs as programmable functions"',
            xy=(0.71, 0.65), xytext=(0.71, 0.92),
            ha='center', fontsize=9, color='#336666', fontweight='bold',
            arrowprops=dict(arrowstyle='-[, widthB=6.0',
                            color='#336666', lw=1.5))

ax.set_xlim(-0.15, 1.15)
ax.set_ylim(-0.05, 1.1)
ax.axis('off')

plt.tight_layout()
plt.show()

print("Key: DSPy and RLM both treat LLMs as programmable functions.")
print("     DSPy gives the developer the programming tools.")
print("     RLM lets the model program itself.")

## Deeper Comparison: How Each Approach Handles Complexity

Let us trace what happens when a task becomes more complex — say, a question
requiring 4 hops instead of 2:

### Manual Prompting
You would need to rewrite the prompt entirely for each new hop structure.

### DSPy
You add another module call in your pipeline:
```python
class FourHopQA(dspy.Module):
    def __init__(self):
        self.hop1 = dspy.ChainOfThought(...)  # Must define each hop
        self.hop2 = dspy.ChainOfThought(...)  # at design time
        self.hop3 = dspy.ChainOfThought(...)
        self.hop4 = dspy.ChainOfThought(...)
```
You must know the number of hops *before* deployment.

### RLM
The model dynamically decides how many hops it needs:
```python
# The model might write:
facts = []
remaining = question
while remaining:
    fact = llm_query(f"Find one fact: {remaining}")
    facts.append(fact)
    remaining = check_if_more_hops_needed(facts)
FINAL(synthesize(facts))
```
The number of hops emerges from the task itself.

In [ ]:
# ---------------------------------------------------------------------------
# Visualization: Scaling Behaviour
# ---------------------------------------------------------------------------
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# --- Left: Developer effort vs task complexity ---
complexity = [1, 2, 3, 4, 5, 6]
prompt_effort  = [1.0, 2.5, 5.0, 8.0, 12.0, 17.0]  # Quadratic — rewrite each time
dspy_effort    = [1.0, 1.5, 2.0, 2.5, 3.0, 3.5]     # Linear — add a module
rlm_effort     = [1.0, 1.0, 1.0, 1.0, 1.0, 1.0]     # Constant — model handles it

ax1.plot(complexity, prompt_effort, 'o-', color='#FF6B6B', linewidth=2,
         markersize=8, label='Manual Prompting')
ax1.plot(complexity, dspy_effort, 's-', color='#4ECDC4', linewidth=2,
         markersize=8, label='DSPy')
ax1.plot(complexity, rlm_effort, '^-', color='#45B7D1', linewidth=2,
         markersize=8, label='RLM')

ax1.set_xlabel('Task Complexity (number of reasoning hops)', fontsize=11)
ax1.set_ylabel('Developer Effort (relative)', fontsize=11)
ax1.set_title('Developer Effort vs Task Complexity', fontsize=13)
ax1.legend(fontsize=10)
ax1.grid(True, alpha=0.3)

# --- Right: Predictability vs complexity ---
prompt_pred  = [0.95, 0.90, 0.80, 0.65, 0.50, 0.35]  # Degrades as prompts get unwieldy
dspy_pred    = [0.95, 0.93, 0.90, 0.88, 0.85, 0.82]  # Stays high — structured pipeline
rlm_pred     = [0.80, 0.75, 0.70, 0.65, 0.60, 0.55]  # Model code varies per run

ax2.plot(complexity, prompt_pred, 'o-', color='#FF6B6B', linewidth=2,
         markersize=8, label='Manual Prompting')
ax2.plot(complexity, dspy_pred, 's-', color='#4ECDC4', linewidth=2,
         markersize=8, label='DSPy')
ax2.plot(complexity, rlm_pred, '^-', color='#45B7D1', linewidth=2,
         markersize=8, label='RLM')

ax2.set_xlabel('Task Complexity (number of reasoning hops)', fontsize=11)
ax2.set_ylabel('Output Predictability (0-1)', fontsize=11)
ax2.set_title('Predictability vs Task Complexity', fontsize=13)
ax2.legend(fontsize=10)
ax2.grid(True, alpha=0.3)
ax2.set_ylim(0, 1.05)

plt.tight_layout()
plt.show()

print("Left:  RLM requires no extra developer effort as tasks get more complex.")
print("Right: DSPy maintains the highest predictability due to its fixed structure.")
print("       RLM trades predictability for adaptability.")

## Exercise: Build a DSPy Pipeline and Compare with RLM

Below we build a DSPy-style pipeline for a summarize-then-answer task and compare
it with the RLM approach on the same question.

**Task**: Given all multi-hop documents, first summarize each one, then answer
a question that requires information from the summaries.

In [ ]:
# ---------------------------------------------------------------------------
# Exercise: DSPy pipeline — Summarize + Answer
# ---------------------------------------------------------------------------

# DSPy approach: developer defines a 2-stage pipeline
class SimulatedSummarizeThenAnswer:
    """DSPy-style pipeline: summarize each doc, then answer from summaries.
    
    Real DSPy equivalent:
        class SummarizeThenAnswer(dspy.Module):
            def __init__(self):
                self.summarize = dspy.ChainOfThought('document -> summary')
                self.answer = dspy.ChainOfThought('summaries, question -> answer')
            
            def forward(self, documents, question):
                summaries = [self.summarize(document=d).summary for d in documents]
                return self.answer(summaries='\n'.join(summaries), question=question)
    """
    
    def __init__(self, lm):
        self.lm = lm
        self.call_count = 0
    
    def forward(self, documents, question):
        # Stage 1: Summarize each document
        summaries = []
        for i, doc in enumerate(documents):
            self.lm(f"Summarize: {doc[:200]}")
            self.call_count += 1
            # Simulated summaries
            if "TechVista Inc was founded" in doc:
                summaries.append("TechVista Inc: founded 2019 by Maria Chen in Austin, TX.")
            elif "NovaPad" in doc:
                summaries.append("NovaPad: flagship tablet by TechVista, 12.4-inch OLED.")
            elif "Dr. James Park" in doc:
                summaries.append("Dr. James Park: lead engineer at TechVista, PhD from Georgia Tech.")
            else:
                summaries.append(f"Document {i+1}: technology-related content.")
        
        # Stage 2: Answer from summaries
        combined = "\n".join(summaries)
        response = self.lm(f"Summaries:\n{combined}\nQuestion: {question}")[0]
        self.call_count += 1
        
        return SimulatedPrediction(
            summaries=summaries,
            answer="Dr. James Park"
        )


# Run DSPy pipeline
exercise_q = "Who is the lead engineer at the company that developed the NovaPad?"

dspy_pipeline = SimulatedSummarizeThenAnswer(dspy_lm)
dspy_ex_result = dspy_pipeline.forward(documents=docs[:3], question=exercise_q)

print("=== DSPy Pipeline: Summarize-then-Answer ===")
print("Stage 1 — Summaries:")
for i, s in enumerate(dspy_ex_result.summaries):
    print(f"  Doc {i+1}: {s}")
print(f"\nStage 2 — Answer: {dspy_ex_result.answer}")
print(f"Total LM calls: {dspy_pipeline.call_count} ({len(docs[:3])} summaries + 1 answer)")
print()

# Run RLM
rlm_ex_result = engine.run(exercise_q, multi_context)
print("=== RLM: Self-Programmed ===")
print(f"Answer: {rlm_ex_result.answer}")
print(f"LLM calls: {rlm_ex_result.total_llm_calls}")
print(f"Strategy: Model wrote code to regex-search across all docs at once")
print()
print("Observation:")
print(f"  DSPy used {dspy_pipeline.call_count} LLM calls (fixed pipeline structure).")
print(f"  RLM used {rlm_ex_result.total_llm_calls} LLM call(s) (model chose its own strategy).")
print("  DSPy's pipeline is more transparent; RLM's approach is more flexible.")

## When to Use What?

| Scenario | Best Approach | Why |
|----------|--------------|-----|
| Well-defined task with training data | **DSPy** | Optimizer can tune prompts on your data |
| Production system needing reliability | **DSPy** | Fixed pipeline structure is predictable |
| Novel / open-ended tasks | **RLM** | Model adapts its strategy per input |
| Tasks with variable complexity | **RLM** | Model decides how many hops / steps needed |
| Rapid prototyping | **DSPy** | Quick to compose from existing modules |
| Research / exploring model capabilities | **RLM** | See what strategies the model invents |
| Hybrid: structured with escape hatch | **DSPy + RLM** | Use DSPy for the pipeline, RLM for hard sub-tasks |

In [ ]:
# ---------------------------------------------------------------------------
# Summary Architecture Diagram
# ---------------------------------------------------------------------------
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

fig, axes = plt.subplots(1, 3, figsize=(16, 6))

# --- Column 1: Manual Prompting ---
ax = axes[0]
ax.set_title('Manual Prompting', fontsize=13, fontweight='bold', color='#FF6B6B')
boxes = [
    (0.5, 0.82, 'Developer writes\nprompt by hand', '#FF6B6B'),
    (0.5, 0.50, 'LLM reads prompt\nand responds', '#FFD93D'),
    (0.5, 0.18, 'Single output', '#DDDDDD'),
]
for x, y, txt, color in boxes:
    ax.add_patch(mpatches.FancyBboxPatch((x-0.3, y-0.1), 0.6, 0.2,
                 boxstyle='round,pad=0.02', facecolor=color, edgecolor='black'))
    ax.text(x, y, txt, ha='center', va='center', fontsize=9)
ax.annotate('', xy=(0.5, 0.62), xytext=(0.5, 0.72),
            arrowprops=dict(arrowstyle='->', color='black', lw=1.5))
ax.annotate('', xy=(0.5, 0.30), xytext=(0.5, 0.40),
            arrowprops=dict(arrowstyle='->', color='black', lw=1.5))
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.axis('off')

# --- Column 2: DSPy ---
ax = axes[1]
ax.set_title('DSPy', fontsize=13, fontweight='bold', color='#4ECDC4')
boxes = [
    (0.5, 0.88, 'Developer defines\nSignatures + Modules', '#4ECDC4'),
    (0.5, 0.68, 'Optimizer tunes\nprompts & demos', '#66DDB8'),
    (0.25, 0.44, 'Module 1\n(CoT)', '#FFD93D'),
    (0.75, 0.44, 'Module 2\n(CoT)', '#FFD93D'),
    (0.5, 0.18, 'Structured\noutput', '#DDDDDD'),
]
for x, y, txt, color in boxes:
    w = 0.35 if 'Module' in txt else 0.6
    ax.add_patch(mpatches.FancyBboxPatch((x-w/2, y-0.1), w, 0.2,
                 boxstyle='round,pad=0.02', facecolor=color, edgecolor='black'))
    ax.text(x, y, txt, ha='center', va='center', fontsize=9)
ax.annotate('', xy=(0.5, 0.60), xytext=(0.5, 0.78),
            arrowprops=dict(arrowstyle='->', color='black', lw=1.5))
ax.annotate('', xy=(0.25, 0.56), xytext=(0.35, 0.60),
            arrowprops=dict(arrowstyle='->', color='black', lw=1.5))
ax.annotate('', xy=(0.75, 0.56), xytext=(0.65, 0.60),
            arrowprops=dict(arrowstyle='->', color='black', lw=1.5))
ax.annotate('', xy=(0.5, 0.30), xytext=(0.25, 0.34),
            arrowprops=dict(arrowstyle='->', color='black', lw=1.5))
ax.annotate('', xy=(0.5, 0.30), xytext=(0.75, 0.34),
            arrowprops=dict(arrowstyle='->', color='black', lw=1.5))
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.axis('off')

# --- Column 3: RLM ---
ax = axes[2]
ax.set_title('RLM', fontsize=13, fontweight='bold', color='#45B7D1')
boxes = [
    (0.5, 0.88, 'Developer provides\ntask + sandbox', '#45B7D1'),
    (0.5, 0.65, 'Model writes\ncode', '#FFD93D'),
    (0.2, 0.42, 'regex\nsearch', '#88CCEE'),
    (0.5, 0.42, 'sub-call\nllm_query()', '#88CCEE'),
    (0.8, 0.42, 'loop /\nbranch', '#88CCEE'),
    (0.5, 0.18, 'FINAL(answer)', '#DDDDDD'),
]
for x, y, txt, color in boxes:
    w = 0.3 if y == 0.42 else 0.6
    h = 0.15 if y == 0.42 else 0.17
    ax.add_patch(mpatches.FancyBboxPatch((x-w/2, y-h/2), w, h,
                 boxstyle='round,pad=0.02', facecolor=color, edgecolor='black'))
    ax.text(x, y, txt, ha='center', va='center', fontsize=8)
ax.annotate('', xy=(0.5, 0.75), xytext=(0.5, 0.80),
            arrowprops=dict(arrowstyle='->', color='black', lw=1.5))
for xp in [0.2, 0.5, 0.8]:
    ax.annotate('', xy=(xp, 0.50), xytext=(0.5, 0.57),
                arrowprops=dict(arrowstyle='->', color='black', lw=1))
for xp in [0.2, 0.5, 0.8]:
    ax.annotate('', xy=(0.5, 0.27), xytext=(xp, 0.35),
                arrowprops=dict(arrowstyle='->', color='black', lw=1))
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.axis('off')

plt.suptitle('Architecture Comparison: Three Paradigms for LLM Programming',
             fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## Key Takeaways

1. **DSPy and RLM are complementary**, not competing. DSPy optimizes structured LLM programs; RLM lets models write their own programs at inference time.

2. **Omar Khattab co-authored both** DSPy and the RLM paper. The shared insight is that LLMs should be treated as programmable functions, not chatbots.

3. **DSPy's core abstractions** — Signatures (I/O specs), Modules (composable programs), and Optimizers (auto-tuning) — bring software engineering discipline to LLM pipelines.

4. **RLM's core idea** — the model writes code in a sandbox and can recursively call itself — gives maximum flexibility but less predictability.

5. **The autonomy spectrum** runs from manual prompting (full developer control) through DSPy (structured modules with optimized prompts) to RLM (self-programming). Each step trades developer control for model autonomy.

6. **Use DSPy** when you have well-defined tasks, training data for optimization, and need production reliability.

7. **Use RLM** when tasks are open-ended, have variable complexity, or you want to explore what strategies the model invents.

8. **The future may combine both**: a DSPy pipeline that uses RLM as a module for sub-tasks that require dynamic reasoning.

---

This concludes the **Comparisons Track** (Notebooks 9-11):
- **Notebook 9**: RAG retrieves context; RLM reasons over it. Use RAG for simple lookups, RLM for aggregation and multi-hop.
- **Notebook 10**: ReAct acts one step at a time; RLM writes a complete program. RLM is more efficient but less transparent.
- **Notebook 11** (this one): DSPy composes predefined modules; RLM composes dynamically. Both treat LLMs as programmable functions.

The common thread: **the more autonomy you give the model, the more capable but less predictable it becomes.** The art is choosing the right point on the spectrum for your use case.